# 🧹 Fake News Detection — Dataset Cleaning Pipeline

This notebook cleans and normalises **all raw datasets** into a single unified schema
ready for JSON preprocessing and model training.

---

## 🎯 Target Schema (`Dataset_Clean.csv`)

| Column | Type | Description |
|---|---|---|
| `id` | str | Unique row identifier |
| `title` | str | Headline / short statement |
| `content` | str | Full article body text (mirrors title if no body) |
| `label` | int | **0=True, 1=Fake, 2=Satire, 3=Bias** |
| `label_text` | str | Human-readable label |
| `label_original` | str | Raw label from source dataset |
| `source_dataset` | str | Which dataset this row came from |
| `topic` | str | Subject / category tag (if available) |
| `url` | str | Source URL (if available) |
| `speaker` | str | Named speaker / author (if available) |

---

## 📦 Datasets Covered

| # | Dataset | File(s) | Labels |
|---|---|---|---|
| 1 | **LIAR** | `LIAR_train/valid/test.tsv` | 6-class → 4-class mapping |
| 2 | **MultiClass2** | `MultiClass2_train/test.csv` | Numeric LIAR — train deduplicated, test saved as holdout |
| 3 | **MultiClass1** | `MultiClass1.csv` | PolitiFact headlines (latin-1) |
| 4 | **BuzzFeed** | `BuzzFeed_fake/real_news_content.csv` | Filename-derived |
| 5 | **PolitiFact Scraped** | `PolitiFact_fake/real_news_content.csv` | Filename-derived |
| 6 | **The Onion** | `The_Onions_Breaking_News_satire.csv` | Satire (2) |
| 7 | **NotTheOnion** | `nottheonion_satire.csv` | True (0) — real absurd headlines |
| 8 | **Propaganda** | `propaganda_dataset.csv` | 0=True, 1=Bias (3) |
| 9 | **India MythFacts** | `India_MythFacts.csv` | 0=Fake (1), 1=True (0) |
| 10 | **Twitter** | `Twitter Raw Data.csv` | majority_target TRUE/FALSE |
| 11 | **ISOT** | `ISOT Fake.csv` + `ISOT True.csv` | Filename-derived |

---

## 🏷️ Label Mapping

| Original | Class |
|---|---|
| `true`, `mostly-true`, `TRUE`, `no-flip`, propaganda=0, myth=False, ISOT_real, twitter=TRUE | **0 — True** |
| `false`, `pants-fire`, `FALSE`, myth=True, ISOT_fake, twitter=FALSE | **1 — Fake** |
| The Onion articles | **2 — Satire** |
| `barely-true`, `half-true`, `half-flip`, `full-flop`, propaganda=1 | **3 — Bias** |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Cell 0 — Prerequisites & Setup

In [2]:
# ── Prerequisites ──────────────────────────────────────────────────────────────
# Standard library only — no extra installs needed for these datasets.
# Uncomment if pandas/numpy are missing:
#   !pip install pandas numpy

import pandas as pd
import numpy as np
import re, os, warnings
warnings.filterwarnings('ignore')

# ── Config ─────────────────────────────────────────────────────────────────────
# Set DATA_DIR to the folder that contains all raw datasets AND this notebook.
DATA_DIR   = os.getcwd()
OUTPUT_CSV = os.path.join(DATA_DIR, 'Dataset_Clean.csv')

# ── Unified label map ──────────────────────────────────────────────────────────
LABEL_INT  = {0: 'True', 1: 'Fake', 2: 'Satire', 3: 'Bias'}

# 6-class LIAR / PolitiFact → 4-class
LIAR_LABEL_MAP = {
    'true'        : 0,   # True
    'mostly-true' : 0,   # True
    'TRUE'        : 0,   # True  (MultiClass1 casing)
    'no-flip'     : 0,   # True  (politician stayed consistent)
    'false'       : 1,   # Fake
    'pants-fire'  : 1,   # Fake
    'FALSE'       : 1,   # Fake  (MultiClass1 casing)
    'barely-true' : 3,   # Bias
    'half-true'   : 3,   # Bias
    'half-flip'   : 3,   # Bias  (partial position reversal)
    'full-flop'   : 3,   # Bias  (complete position reversal)
}

TARGET_COLS = ['id','title','content','label','label_text',
               'label_original','source_dataset','topic','url','speaker']

# ── Shared helpers ─────────────────────────────────────────────────────────────
def make_id(prefix: str, idx: int) -> str:
    return f"{prefix}_{str(idx).zfill(6)}"

def clean_text(text) -> str:
    """Strip HTML, URLs, smart-quotes, collapse whitespace."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Accumulator — each cleaned dataset is appended here before final merge
all_frames = []

print('✅  Setup complete.')
print(f'   Output  : {OUTPUT_CSV}')
print(f'   Labels  : {LABEL_INT}')

✅  Setup complete.
   Output  : /content/Dataset_Clean.csv
   Labels  : {0: 'True', 1: 'Fake', 2: 'Satire', 3: 'Bias'}


---
## Cell 1 — LIAR Dataset
**Files:** `LIAR_train.tsv`, `LIAR_valid.tsv`, `LIAR_test.tsv`  
**Format:** Tab-separated, no header, 14 columns  
**Labels:** 6-class string → 4-class mapping  
**Content:** Short political statements (no full article body)

In [5]:
# ── LIAR Dataset ───────────────────────────────────────────────────────────────
# Col positions (no header row):
#  0 raw_id  1 label  2 statement  3 subject  4 speaker  5 speaker_job
#  6 state   7 party  8-12 historical counts  13 context

LIAR_COLS = [
    'raw_id','label_original','statement','subject',
    'speaker','speaker_job','state','party',
    'cnt_barely_true','cnt_false','cnt_half_true',
    'cnt_mostly_true','cnt_pants_fire','context'
]

liar_splits = {'train':'LIAR_train.tsv', 'valid':'LIAR_valid.tsv', 'test':'LIAR_test.tsv'}
liar_frames = []
for split, fname in liar_splits.items():
    path = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(path, sep='\t', header=None, names=LIAR_COLS, dtype=str)
    df['_split'] = split
    liar_frames.append(df)
    print(f'  Loaded {fname}: {len(df):,} rows')

liar = pd.concat(liar_frames, ignore_index=True)
liar['label_original'] = liar['label_original'].str.strip()

# Drop rows with unmappable labels
unmapped = liar[~liar['label_original'].isin(LIAR_LABEL_MAP)]
if len(unmapped):
    print(f'  ⚠️  Dropping {len(unmapped)} unmapped labels: {unmapped["label_original"].unique()}')
liar = liar[liar['label_original'].isin(LIAR_LABEL_MAP)].copy()

liar_clean = pd.DataFrame()
liar_clean['id']             = [make_id('LIAR', i) for i in range(len(liar))]
liar_clean['title']          = liar['statement'].apply(clean_text).values
liar_clean['content']        = liar['statement'].apply(clean_text).values
liar_clean['label']          = liar['label_original'].map(LIAR_LABEL_MAP).values
liar_clean['label_text']     = liar_clean['label'].map(LABEL_INT).values
liar_clean['label_original'] = liar['label_original'].values
liar_clean['source_dataset'] = 'LIAR'
liar_clean['topic']          = liar['subject'].fillna('').apply(clean_text).values
liar_clean['url']            = ''
liar_clean['speaker']        = liar['speaker'].fillna('').apply(clean_text).values

print('\n  Label distribution:')
print(liar_clean['label_text'].value_counts().to_string())
all_frames.append(liar_clean)
print(f'\n✅  LIAR done — {len(liar_clean):,} rows added.')

  Loaded LIAR_train.tsv: 10,240 rows
  Loaded LIAR_valid.tsv: 1,284 rows
  Loaded LIAR_test.tsv: 1,267 rows

  Label distribution:
label_text
Bias    4730
True    4507
Fake    3554

✅  LIAR done — 12,791 rows added.


---
## Cell 2 — MultiClass2 Dataset
**Files:** `MultiClass2_train.csv`, `MultiClass2_test.csv`  
**Note:** `MultiClass2_train` is an **identical numeric re-encoding** of `LIAR_train` — skipped to avoid duplicates.  
`MultiClass2_test` is **unlabelled** — saved separately as `Dataset_Unlabelled_Holdout.csv`.

In [6]:
# ── MultiClass2 ────────────────────────────────────────────────────────────────
# Train = numeric LIAR_train (skip); Test = unlabelled holdout (save separately)

mc2_train = pd.read_csv(os.path.join(DATA_DIR, 'MultiClass2_train.csv'))
liar_ref   = pd.read_csv(os.path.join(DATA_DIR, 'LIAR_train.tsv'),
                          sep='\t', header=None, dtype=str)
is_dup = (mc2_train['Text'].iloc[0].strip() == liar_ref.iloc[0, 2].strip())
print(f'  MultiClass2_train == LIAR_train: {is_dup}  →  skipping {len(mc2_train):,} duplicate rows.')

# Save unlabelled test rows as separate holdout file
mc2_test = pd.read_csv(os.path.join(DATA_DIR, 'MultiClass2_test.csv'))
holdout = pd.DataFrame()
holdout['id']             = [make_id('MC2TEST', i) for i in range(len(mc2_test))]
holdout['title']          = mc2_test['Text'].apply(clean_text).values
holdout['content']        = mc2_test['Text'].apply(clean_text).values
holdout['label']          = np.nan
holdout['label_text']     = 'UNLABELLED'
holdout['label_original'] = 'UNLABELLED'
holdout['source_dataset'] = 'MultiClass2_Test'
holdout['topic']          = mc2_test['Text_Tag'].fillna('').apply(clean_text).values
holdout['url']            = ''
holdout['speaker']        = ''

holdout_path = os.path.join(DATA_DIR, 'Dataset_Unlabelled_Holdout.csv')
holdout.to_csv(holdout_path, index=False, encoding='utf-8')
print(f'  ✅  Unlabelled holdout saved → Dataset_Unlabelled_Holdout.csv ({len(holdout):,} rows)')
print(f'  ℹ️   NOT added to Dataset_Clean.csv — no ground-truth label.')

  MultiClass2_train == LIAR_train: True  →  skipping 10,240 duplicate rows.
  ✅  Unlabelled holdout saved → Dataset_Unlabelled_Holdout.csv (1,267 rows)
  ℹ️   NOT added to Dataset_Clean.csv — no ground-truth label.


---
## Cell 3 — MultiClass1 Dataset
**File:** `MultiClass1.csv` (latin-1 encoded)  
**Columns:** `News_Headline`, `Link_Of_News`, `Source`, `Stated_On`, `Date`, `Label`  
**Labels:** PolitiFact-style 6-point scale → 4-class mapping  
**Content:** Headlines only — `content` mirrors `title`

In [7]:
# ── MultiClass1 ────────────────────────────────────────────────────────────────
df_mc1 = pd.read_csv(os.path.join(DATA_DIR, 'MultiClass1.csv'), encoding='latin-1')
print(f'  Loaded: {len(df_mc1):,} rows')
print('  Raw label distribution:')
print(df_mc1['Label'].value_counts().to_string())

df_mc1['Label'] = df_mc1['Label'].astype(str).str.strip()
unmapped = df_mc1[~df_mc1['Label'].isin(LIAR_LABEL_MAP)]
if len(unmapped):
    print(f'\n  ⚠️  Dropping {len(unmapped)} unmappable rows: {unmapped["Label"].unique()}')
df_mc1 = df_mc1[df_mc1['Label'].isin(LIAR_LABEL_MAP)].copy().reset_index(drop=True)

mc1_clean = pd.DataFrame()
mc1_clean['id']             = [make_id('MC1', i) for i in range(len(df_mc1))]
mc1_clean['title']          = df_mc1['News_Headline'].apply(clean_text).values
mc1_clean['content']        = df_mc1['News_Headline'].apply(clean_text).values
mc1_clean['label']          = df_mc1['Label'].map(LIAR_LABEL_MAP).values
mc1_clean['label_text']     = mc1_clean['label'].map(LABEL_INT).values
mc1_clean['label_original'] = df_mc1['Label'].values
mc1_clean['source_dataset'] = 'MultiClass1'
mc1_clean['topic']          = ''
mc1_clean['url']            = df_mc1['Link_Of_News'].fillna('').values
mc1_clean['speaker']        = df_mc1['Source'].fillna('').apply(clean_text).values

print('\n  Label distribution after normalisation:')
print(mc1_clean['label_text'].value_counts().to_string())
all_frames.append(mc1_clean)
print(f'\n✅  MultiClass1 done — {len(mc1_clean):,} rows added.')

  Loaded: 9,960 rows
  Raw label distribution:
Label
FALSE          2273
barely-true    1737
mostly-true    1722
half-true      1685
pants-fire     1402
TRUE           1036
full-flop        70
half-flip        27
no-flip           8

  Label distribution after normalisation:
label_text
Fake    3675
Bias    3519
True    2766

✅  MultiClass1 done — 9,960 rows added.


---
## Cell 4 — BuzzFeed Dataset
**Files:** `BuzzFeed_fake_news_content.csv`, `BuzzFeed_real_news_content.csv`  
**Label:** Derived from filename — `fake` → 1, `real` → 0  
**Dropped:** `meta_data`, `images`, `movies`, `top_img`, `canonical_link`

In [8]:
# ── BuzzFeed Dataset ───────────────────────────────────────────────────────────
KEEP_COLS = ['id','title','text','url','authors','source','publish_date']
buzzfeed_files = {
    'BuzzFeed_fake_news_content.csv': (1, 'fake'),
    'BuzzFeed_real_news_content.csv': (0, 'real'),
}
bf_frames = []
for fname, (li, ls) in buzzfeed_files.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname), usecols=lambda c: c in KEEP_COLS)
    df['_li'] = li; df['_ls'] = ls
    print(f'  Loaded {fname}: {len(df):,} rows')
    bf_frames.append(df)

df_bf = pd.concat(bf_frames, ignore_index=True)
df_bf['text']  = df_bf['text'].fillna('')
df_bf['title'] = df_bf['title'].fillna('')
df_bf = df_bf[(df_bf['text'].str.len() > 10) | (df_bf['title'].str.len() > 5)].copy()
df_bf.reset_index(drop=True, inplace=True)

bf_clean = pd.DataFrame()
bf_clean['id']             = [make_id('BF', i) for i in range(len(df_bf))]
bf_clean['title']          = df_bf['title'].apply(clean_text).values
bf_clean['content']        = df_bf['text'].apply(clean_text).values
bf_clean['label']          = df_bf['_li'].values
bf_clean['label_text']     = df_bf['_li'].map(LABEL_INT).values
bf_clean['label_original'] = df_bf['_ls'].values
bf_clean['source_dataset'] = 'BuzzFeed'
bf_clean['topic']          = ''
bf_clean['url']            = df_bf['url'].fillna('').values
bf_clean['speaker']        = df_bf['authors'].fillna('').apply(clean_text).values

print('\n  Label distribution:')
print(bf_clean['label_text'].value_counts().to_string())
all_frames.append(bf_clean)
print(f'\n✅  BuzzFeed done — {len(bf_clean):,} rows added.')

  Loaded BuzzFeed_fake_news_content.csv: 91 rows
  Loaded BuzzFeed_real_news_content.csv: 91 rows

  Label distribution:
label_text
Fake    91
True    91

✅  BuzzFeed done — 182 rows added.


---
## Cell 5 — PolitiFact Scraped Dataset
**Files:** `PolitiFact_fake_news_content.csv`, `PolitiFact_real_news_content.csv`  
**Label:** Derived from filename  
**Note:** Cross-deduplication against BuzzFeed URLs (both from FakeNewsNet)

In [9]:
# ── PolitiFact Scraped Dataset ─────────────────────────────────────────────────
pf_files = {
    'PolitiFact_fake_news_content.csv': (1, 'fake'),
    'PolitiFact_real_news_content.csv': (0, 'real'),
}
pf_frames = []
for fname, (li, ls) in pf_files.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname), usecols=lambda c: c in KEEP_COLS)
    df['_li'] = li; df['_ls'] = ls
    print(f'  Loaded {fname}: {len(df):,} rows')
    pf_frames.append(df)

df_pf = pd.concat(pf_frames, ignore_index=True)
df_pf['text']  = df_pf['text'].fillna('')
df_pf['title'] = df_pf['title'].fillna('')
df_pf = df_pf[(df_pf['text'].str.len() > 10) | (df_pf['title'].str.len() > 5)].copy()

# Cross-dedup against BuzzFeed
bf_urls = set(bf_clean['url'].dropna())
before  = len(df_pf)
df_pf   = df_pf[~df_pf['url'].isin(bf_urls)].copy().reset_index(drop=True)
print(f'  Removed {before - len(df_pf)} rows overlapping with BuzzFeed URLs.')

pf_clean = pd.DataFrame()
pf_clean['id']             = [make_id('PF', i) for i in range(len(df_pf))]
pf_clean['title']          = df_pf['title'].apply(clean_text).values
pf_clean['content']        = df_pf['text'].apply(clean_text).values
pf_clean['label']          = df_pf['_li'].values
pf_clean['label_text']     = df_pf['_li'].map(LABEL_INT).values
pf_clean['label_original'] = df_pf['_ls'].values
pf_clean['source_dataset'] = 'PolitiFact_Scraped'
pf_clean['topic']          = ''
pf_clean['url']            = df_pf['url'].fillna('').values
pf_clean['speaker']        = df_pf['authors'].fillna('').apply(clean_text).values

print('\n  Label distribution:')
print(pf_clean['label_text'].value_counts().to_string())
all_frames.append(pf_clean)
print(f'\n✅  PolitiFact Scraped done — {len(pf_clean):,} rows added.')

  Loaded PolitiFact_fake_news_content.csv: 120 rows
  Loaded PolitiFact_real_news_content.csv: 120 rows
  Removed 2 rows overlapping with BuzzFeed URLs.

  Label distribution:
label_text
Fake    119
True    119

✅  PolitiFact Scraped done — 238 rows added.


---
## Cell 6 — The Onion (Satire)
**File:** `The_Onions_Breaking_News_satire.csv`  
**Columns:** `Title`, `Published Time`, `Content`  
**Label:** **2 — Satire** (all rows)  
**Note:** This is the primary Satire source. Full article content is available.

In [10]:
# ── The Onion Satire ───────────────────────────────────────────────────────────
df_onion = pd.read_csv(os.path.join(DATA_DIR, 'The_Onions_Breaking_News_satire.csv'))
print(f'  Loaded: {len(df_onion):,} rows')
print(f'  Columns: {list(df_onion.columns)}')
print(f'  Null Content: {df_onion["Content"].isna().sum()}')
print(f'  Avg content length: {df_onion["Content"].str.len().mean():.0f} chars')

# Drop rows with no content and no title
df_onion['Title']   = df_onion['Title'].fillna('')
df_onion['Content'] = df_onion['Content'].fillna('')
df_onion = df_onion[
    (df_onion['Title'].str.len() > 5) | (df_onion['Content'].str.len() > 10)
].copy().reset_index(drop=True)

onion_clean = pd.DataFrame()
onion_clean['id']             = [make_id('ONION', i) for i in range(len(df_onion))]
onion_clean['title']          = df_onion['Title'].apply(clean_text).values
onion_clean['content']        = df_onion['Content'].apply(clean_text).values
onion_clean['label']          = 2   # Satire
onion_clean['label_text']     = 'Satire'
onion_clean['label_original'] = 'satire'
onion_clean['source_dataset'] = 'TheOnion'
onion_clean['topic']          = ''
onion_clean['url']            = ''
onion_clean['speaker']        = ''

print('\n  Label distribution:')
print(onion_clean['label_text'].value_counts().to_string())
print(f'  Sample title: {onion_clean["title"].iloc[0]}')
all_frames.append(onion_clean)
print(f'\n✅  The Onion done — {len(onion_clean):,} rows added.')

  Loaded: 6,851 rows
  Columns: ['Title', 'Published Time', 'Content']
  Null Content: 5
  Avg content length: 1018 chars

  Label distribution:
label_text
Satire    6851
  Sample title: Report: Iran Less Than 10 Years Away From 2016

✅  The Onion done — 6,851 rows added.


---
## Cell 7 — r/NotTheOnion (Real Absurd News)
**File:** `nottheonion_satire.csv`  
**Source:** Reddit r/NotTheOnion — real news headlines so absurd they *sound* satirical  
**Label:** **0 — True** ⚠️ Despite the filename containing "satire", these are **genuine real news stories**.  
The subreddit posts real articles that seem fake. `title` is the headline; `body` is mostly null (39% missing).  
**Design decision:** Labelled True because they are factually true events — their absurdity makes them  
valuable training data that teaches the model to distinguish bizarre-but-real news from actual satire.

In [11]:
# ── r/NotTheOnion — Real Absurd Headlines ──────────────────────────────────────
# ⚠️  Despite the filename, these are REAL news stories (not satire).
# r/NotTheOnion = real news so absurd it sounds fake → label as True (0)

df_nto = pd.read_csv(os.path.join(DATA_DIR, 'nottheonion_satire.csv'))
print(f'  Loaded: {len(df_nto):,} rows')
print(f'  Columns: {list(df_nto.columns)}')
print(f'  Body null: {df_nto["body"].isna().sum()} ({df_nto["body"].isna().mean()*100:.0f}%)')

df_nto['title'] = df_nto['title'].fillna('')
df_nto['body']  = df_nto['body'].fillna('')
df_nto = df_nto[df_nto['title'].str.len() > 5].copy().reset_index(drop=True)

nto_clean = pd.DataFrame()
nto_clean['id']             = [make_id('NTO', i) for i in range(len(df_nto))]
nto_clean['title']          = df_nto['title'].apply(clean_text).values
# Use body as content where available, else fall back to title
nto_clean['content']        = df_nto.apply(
    lambda r: clean_text(r['body']) if len(str(r['body'])) > 10 else clean_text(r['title']),
    axis=1
).values
nto_clean['label']          = 0   # True — real news (absurd but factual)
nto_clean['label_text']     = 'True'
nto_clean['label_original'] = 'real_absurd'
nto_clean['source_dataset'] = 'NotTheOnion_Reddit'
nto_clean['topic']          = ''
nto_clean['url']            = df_nto['url'].fillna('').values
nto_clean['speaker']        = ''

print('\n  Label distribution:')
print(nto_clean['label_text'].value_counts().to_string())
print(f'  Sample title: {nto_clean["title"].iloc[0]}')
all_frames.append(nto_clean)
print(f'\n✅  NotTheOnion done — {len(nto_clean):,} rows added.')

  Loaded: 1,627 rows
  Columns: ['title', 'score', 'id', 'url', 'comms_num', 'created', 'body', 'timestamp']
  Body null: 637 (39%)

  Label distribution:
label_text
True    1627
  Sample title: Human heart found in TDOT salt pile in Humphreys County

✅  NotTheOnion done — 1,627 rows added.


---
## Cell 8 — Propaganda Dataset
**File:** `propaganda_dataset.csv`  
**Columns:** `text`, `label`  
**Labels:**  
- `0` = Non-propaganda (factual, neutral statements) → **0 — True**  
- `1` = Propaganda (manipulative, emotionally charged, divisive) → **3 — Bias**  
**Note:** Short texts (~65 chars avg). 5,000 rows per class (balanced).

In [12]:
# ── Propaganda Dataset ────────────────────────────────────────────────────────
# label=0 → neutral/factual    → True (0)
# label=1 → propaganda/biased  → Bias (3)

df_prop = pd.read_csv(os.path.join(DATA_DIR, 'propaganda_dataset.csv'))
print(f'  Loaded: {len(df_prop):,} rows')
print('  Raw label distribution:')
print(df_prop['label'].value_counts().to_string())

PROP_LABEL_MAP = {0: 0, 1: 3}   # 0=True, 1=Bias
PROP_ORIG_MAP  = {0: 'non_propaganda', 1: 'propaganda'}

df_prop = df_prop.dropna(subset=['text']).copy()
df_prop = df_prop[df_prop['text'].str.len() > 5].copy().reset_index(drop=True)

prop_clean = pd.DataFrame()
prop_clean['id']             = [make_id('PROP', i) for i in range(len(df_prop))]
prop_clean['title']          = df_prop['text'].apply(clean_text).values
prop_clean['content']        = df_prop['text'].apply(clean_text).values
prop_clean['label']          = df_prop['label'].map(PROP_LABEL_MAP).values
prop_clean['label_text']     = prop_clean['label'].map(LABEL_INT).values
prop_clean['label_original'] = df_prop['label'].map(PROP_ORIG_MAP).values
prop_clean['source_dataset'] = 'Propaganda'
prop_clean['topic']          = ''
prop_clean['url']            = ''
prop_clean['speaker']        = ''

print('\n  Label distribution after normalisation:')
print(prop_clean['label_text'].value_counts().to_string())
print(f'  Sample Bias row  : {prop_clean[prop_clean["label"]==3]["title"].iloc[0]}')
print(f'  Sample True row  : {prop_clean[prop_clean["label"]==0]["title"].iloc[0]}')
all_frames.append(prop_clean)
print(f'\n✅  Propaganda done — {len(prop_clean):,} rows added.')

  Loaded: 10,000 rows
  Raw label distribution:
label
1    5000
0    5000

  Label distribution after normalisation:
label_text
Bias    5000
True    5000
  Sample Bias row  : FYI: Look how they twist facts to confuse the people....
  Sample True row  : The hiking trail reopened after seasonal closures., say sources

✅  Propaganda done — 10,000 rows added.


---
## Cell 9 — India MythFacts
**File:** `India_MythFacts.csv`  
**Columns:** `text`, `label`  
**Labels:**  
- `0` = Myth / false popular belief → **1 — Fake**  
- `1` = Verified scientific/health fact → **0 — True**  
**Note:** Small dataset (219 rows) of Indian health/culture myths vs facts. Short statements.

In [13]:
# ── India MythFacts Dataset ───────────────────────────────────────────────────
# label=0 → myth (false belief) → Fake (1)
# label=1 → fact (verified)     → True (0)

df_india = pd.read_csv(os.path.join(DATA_DIR, 'India_MythFacts.csv'))
print(f'  Loaded: {len(df_india):,} rows')
print('  Raw label distribution:')
print(df_india['label'].value_counts().to_string())

INDIA_LABEL_MAP = {0: 1, 1: 0}    # 0=myth→Fake, 1=fact→True
INDIA_ORIG_MAP  = {0: 'myth', 1: 'fact'}

df_india = df_india.dropna(subset=['text']).copy().reset_index(drop=True)

india_clean = pd.DataFrame()
india_clean['id']             = [make_id('IND', i) for i in range(len(df_india))]
india_clean['title']          = df_india['text'].apply(clean_text).values
india_clean['content']        = df_india['text'].apply(clean_text).values
india_clean['label']          = df_india['label'].map(INDIA_LABEL_MAP).values
india_clean['label_text']     = india_clean['label'].map(LABEL_INT).values
india_clean['label_original'] = df_india['label'].map(INDIA_ORIG_MAP).values
india_clean['source_dataset'] = 'India_MythFacts'
india_clean['topic']          = 'health/culture'
india_clean['url']            = ''
india_clean['speaker']        = ''

print('\n  Label distribution after normalisation:')
print(india_clean['label_text'].value_counts().to_string())
print(f'  Sample Fake row : {india_clean[india_clean["label"]==1]["title"].iloc[0]}')
print(f'  Sample True row : {india_clean[india_clean["label"]==0]["title"].iloc[0]}')
all_frames.append(india_clean)
print(f'\n✅  India MythFacts done — {len(india_clean):,} rows added.')

  Loaded: 219 rows
  Raw label distribution:
label
0    119
1    100

  Label distribution after normalisation:
label_text
Fake    119
True    100
  Sample Fake row : "Eating curd at night is bad for health"
  Sample True row : "Vaccines help prevent many infectious diseases"

✅  India MythFacts done — 219 rows added.


---
## Cell 10 — Twitter Raw Data
**File:** `Twitter Raw Data.csv`  
**Key Columns:** `majority_target` (TRUE/FALSE), `statement` (the claim being verified), `tweet` (a related tweet)  
**Labels:**  
- `majority_target = TRUE` → **0 — True**  
- `majority_target = FALSE` → **1 — Fake**  
**Design:** `statement` → `title`, `tweet` → `content`. The statement is the verifiable claim;  
the tweet is a citizen's response to that claim.

In [14]:
# ── Twitter Raw Data ───────────────────────────────────────────────────────────
# File name contains a space — adjust path if needed
TWITTER_FILE = 'Twitter Raw Data.csv'
tw_path = os.path.join(DATA_DIR, TWITTER_FILE)

for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        df_tw = pd.read_csv(tw_path, encoding=enc)
        print(f'  Loaded with encoding={enc}: {len(df_tw):,} rows')
        break
    except UnicodeDecodeError:
        print(f'  {enc} failed, trying next...')

print(f'  Columns: {list(df_tw.columns)}')
print('  majority_target distribution:')
print(df_tw['majority_target'].value_counts().to_string())

# Normalise label: TRUE → 0, FALSE → 1
TWITTER_LABEL_MAP = {'TRUE': 0, 'FALSE': 1}
df_tw['majority_target'] = df_tw['majority_target'].astype(str).str.strip().str.upper()
unmapped_tw = df_tw[~df_tw['majority_target'].isin(TWITTER_LABEL_MAP)]
if len(unmapped_tw):
    print(f'  ⚠️  Dropping {len(unmapped_tw)} rows with unmapped target: {unmapped_tw["majority_target"].unique()}')
df_tw = df_tw[df_tw['majority_target'].isin(TWITTER_LABEL_MAP)].copy()
df_tw['statement'] = df_tw['statement'].fillna('')
df_tw['tweet']     = df_tw['tweet'].fillna('')

# Deduplicate on statement (same claim appears multiple times — one per tweet)
# Keep the row with the longest tweet for maximum content
df_tw['_tw_len'] = df_tw['tweet'].str.len()
df_tw = df_tw.sort_values('_tw_len', ascending=False)
df_tw = df_tw.drop_duplicates(subset='statement', keep='first')
df_tw = df_tw.drop(columns=['_tw_len']).reset_index(drop=True)
print(f'  After dedup on statement: {len(df_tw):,} unique claims')

tw_clean = pd.DataFrame()
tw_clean['id']             = [make_id('TW', i) for i in range(len(df_tw))]
tw_clean['title']          = df_tw['statement'].apply(clean_text).values
tw_clean['content']        = df_tw['tweet'].apply(clean_text).values
tw_clean['label']          = df_tw['majority_target'].map(TWITTER_LABEL_MAP).values
tw_clean['label_text']     = tw_clean['label'].map(LABEL_INT).values
tw_clean['label_original'] = df_tw['majority_target'].values
tw_clean['source_dataset'] = 'Twitter'
tw_clean['topic']          = ''
tw_clean['url']            = ''
tw_clean['speaker']        = df_tw['author'].fillna('').apply(clean_text).values if 'author' in df_tw.columns else ''

print('\n  Label distribution after normalisation:')
print(tw_clean['label_text'].value_counts().to_string())
all_frames.append(tw_clean)
print(f'\n✅  Twitter done — {len(tw_clean):,} rows added.')

  Loaded with encoding=utf-8: 134,198 rows
  Columns: ['Unnamed: 0', 'majority_target', 'statement', 'author', 'tweet', 'followers_count', 'friends_count', 'favourites_count', 'statuses_count', 'listed_count', 'mentions', 'quotes', 'replies', 'retweets', 'favourites', 'hashtags', '5_label_majority_answer', '3_label_majority_answer']
  majority_target distribution:
majority_target
True     68985
False    65213
  After dedup on statement: 1,058 unique claims

  Label distribution after normalisation:
label_text
True    583
Fake    475

✅  Twitter done — 1,058 rows added.


---
## Cell 11 — ISOT Fake News Dataset
**Files:** `ISOT Fake.csv`, `ISOT True.csv`  
**Columns:** `title`, `text`, `subject`, `date`  
**Labels:** Filename-derived — `ISOT Fake.csv` → **1 (Fake)**, `ISOT True.csv` → **0 (True)**  
**Note:** ~44k articles. True = Reuters reporting; Fake = sites flagged by PolitiFact.  
Files use tab-separation or CSV depending on your download — both are handled below.

In [15]:
# ── ISOT Fake News Dataset ────────────────────────────────────────────────────
ISOT_FILES = {
    'ISOT Fake.csv': 1,   # Fake
    'ISOT True.csv': 0,   # True
}

isot_frames = []
for fname, label_int in ISOT_FILES.items():
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f'  ⚠️  {fname} not found — skipping. Place it in {DATA_DIR}')
        continue

    # Try common encodings and separators
    df_isot = None
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        for sep in [',', '\t']:
            try:
                df_isot = pd.read_csv(fpath, encoding=enc, sep=sep)
                if df_isot.shape[1] >= 3:   # must have at least title/text/subject
                    print(f'  Loaded {fname} [enc={enc}, sep={repr(sep)}]: {len(df_isot):,} rows')
                    break
            except Exception:
                pass
        if df_isot is not None and df_isot.shape[1] >= 3:
            break

    if df_isot is None:
        print(f'  ❌  Could not parse {fname} — skipping.')
        continue

    # Normalise column names (handle case differences)
    df_isot.columns = [c.strip().lower() for c in df_isot.columns]
    title_col   = next((c for c in df_isot.columns if 'title'   in c), None)
    text_col    = next((c for c in df_isot.columns if 'text'    in c), None)
    subject_col = next((c for c in df_isot.columns if 'subject' in c), None)

    if not title_col or not text_col:
        print(f'  ❌  Cannot identify title/text columns in {fname}. Columns: {list(df_isot.columns)}')
        continue

    df_isot[text_col]  = df_isot[text_col].fillna('')
    df_isot[title_col] = df_isot[title_col].fillna('')
    df_isot = df_isot[
        (df_isot[text_col].str.len() > 10) | (df_isot[title_col].str.len() > 5)
    ].copy().reset_index(drop=True)

    label_name = 'isot_fake' if label_int == 1 else 'isot_true'
    prefix     = 'ISOTF' if label_int == 1 else 'ISOTT'

    tmp = pd.DataFrame()
    tmp['id']             = [make_id(prefix, i) for i in range(len(df_isot))]
    tmp['title']          = df_isot[title_col].apply(clean_text).values
    tmp['content']        = df_isot[text_col].apply(clean_text).values
    tmp['label']          = label_int
    tmp['label_text']     = LABEL_INT[label_int]
    tmp['label_original'] = label_name
    tmp['source_dataset'] = 'ISOT'
    tmp['topic']          = df_isot[subject_col].fillna('').apply(clean_text).values if subject_col else ''
    tmp['url']            = ''
    tmp['speaker']        = ''
    isot_frames.append(tmp)
    print(f'    → {len(tmp):,} {LABEL_INT[label_int]} rows')

if isot_frames:
    isot_clean = pd.concat(isot_frames, ignore_index=True)
    print('\n  Label distribution:')
    print(isot_clean['label_text'].value_counts().to_string())
    all_frames.append(isot_clean)
    print(f'\n✅  ISOT done — {len(isot_clean):,} rows added.')
else:
    print('\n⚠️  No ISOT files loaded — place ISOT Fake.csv and ISOT True.csv in the DATA_DIR and re-run.')

  Loaded ISOT Fake.csv [enc=utf-8, sep=',']: 23,502 rows
    → 23,502 Fake rows
  Loaded ISOT True.csv [enc=utf-8, sep=',']: 21,417 rows
    → 21,417 True rows

  Label distribution:
label_text
Fake    23502
True    21417

✅  ISOT done — 44,919 rows added.


---
## Cell 13 — Merge All Datasets, Deduplicate & Save

In [16]:
# ── Merge ─────────────────────────────────────────────────────────────────────
print(f'Merging {len(all_frames)} dataset(s)...')
master = pd.concat(all_frames, ignore_index=True)[TARGET_COLS]
print(f'  Rows before dedup : {len(master):,}')

# ── Global content deduplication ──────────────────────────────────────────────
master['_norm'] = (master['content'].str.lower().str.strip()
                   .str.replace(r'\s+', ' ', regex=True))
before  = len(master)
master  = master.drop_duplicates(subset='_norm', keep='first')
master  = master.drop(columns=['_norm'])
print(f'  Removed {before - len(master):,} duplicate content rows.')

# ── Drop rows with no label ────────────────────────────────────────────────────
null_lbl = master['label'].isna()
if null_lbl.sum():
    print(f'  Dropping {null_lbl.sum()} rows with null labels.')
    master = master[~null_lbl].copy()

# ── Drop rows with no text at all ─────────────────────────────────────────────
empty = master['content'].eq('') & master['title'].eq('')
if empty.sum():
    print(f'  Dropping {empty.sum()} completely empty rows.')
    master = master[~empty].copy()

# ── Enforce dtypes ─────────────────────────────────────────────────────────────
master['label'] = master['label'].astype(int)
for col in ['title','content','label_text','label_original',
            'source_dataset','topic','url','speaker']:
    master[col] = master[col].fillna('').astype(str)

master.reset_index(drop=True, inplace=True)

# ── Final stats ────────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('  FINAL DATASET SUMMARY')
print('='*55)
print(f'  Total rows    : {len(master):,}')
print(f'  Total columns : {len(master.columns)}')
print()
print('  Label distribution:')
for li, ln in LABEL_INT.items():
    c = (master['label'] == li).sum()
    pct = c / len(master) * 100
    bar = '█' * int(pct / 2)
    print(f'    {li} {ln:<8}: {c:>7,}  ({pct:5.1f}%)  {bar}')

print()
print('  Source breakdown:')
print(master['source_dataset'].value_counts().to_string())

print()
print('  Empty field check:')
for col in TARGET_COLS:
    n = (master[col].astype(str).str.strip() == '').sum() if col != 'label' else master[col].isna().sum()
    if n > 0:
        flag = '  ⚠️ ' if col in ['title','content'] else '     '
        print(f'{flag}{col}: {n:,} empty')

# ── Save ───────────────────────────────────────────────────────────────────────
master.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f'\n✅  Saved → {OUTPUT_CSV}  ({os.path.getsize(OUTPUT_CSV)/1024/1024:.1f} MB)')

Merging 10 dataset(s)...
  Rows before dedup : 87,845
  Removed 14,119 duplicate content rows.

  FINAL DATASET SUMMARY
  Total rows    : 73,726
  Total columns : 10

  Label distribution:
    0 True    :  32,069  ( 43.5%)  █████████████████████
    1 Fake    :  25,339  ( 34.4%)  █████████████████
    2 Satire  :   6,817  (  9.2%)  ████
    3 Bias    :   9,501  ( 12.9%)  ██████

  Source breakdown:
source_dataset
ISOT                  38595
LIAR                  12765
MultiClass1            9611
TheOnion               6817
Propaganda             2799
NotTheOnion_Reddit     1573
Twitter                1058
India_MythFacts         219
BuzzFeed                178
PolitiFact_Scraped      111

  Empty field check:
  ⚠️ content: 1 empty
     topic: 22,160 empty
     url: 63,203 empty
     speaker: 50,054 empty

✅  Saved → /content/Dataset_Clean.csv  (112.1 MB)


---
## Cell 14 — Quick Preview & Sample Rows

In [17]:
# ── Reload and preview ────────────────────────────────────────────────────────
df_check = pd.read_csv(OUTPUT_CSV)
print(f'Reloaded: {len(df_check):,} rows × {len(df_check.columns)} cols')
print()
print('Column dtypes:')
print(df_check.dtypes.to_string())
print()
print('── One sample per class ──────────────────────────────────────────────────')
for li, ln in LABEL_INT.items():
    subset = df_check[df_check['label'] == li]
    if len(subset) == 0:
        print(f'  {ln} (label={li}): NO ROWS')
        continue
    row = subset.sample(1, random_state=42).iloc[0]
    print(f'\n  ── {ln} (label={li}) ── source: {row["source_dataset"]}')
    print(f'     title  : {str(row["title"])[:110]}')
    content_preview = str(row["content"])[:200]
    print(f'     content: {content_preview}{"..." if len(str(row["content"])) > 200 else ""}')
    if row["topic"]:   print(f'     topic  : {row["topic"]}')
    if row["speaker"]: print(f'     speaker: {row["speaker"]}')

Reloaded: 73,726 rows × 10 cols

Column dtypes:
id                object
title             object
content           object
label              int64
label_text        object
label_original    object
source_dataset    object
topic             object
url               object
speaker           object

── One sample per class ──────────────────────────────────────────────────

  ── True (label=0) ── source: ISOT
     title  : May calls for Brexit talks progress at EU summit
     content: BRUSSELS (Reuters) - Prime Minister Theresa May said she would tell fellow European Union leaders on Thursday of ambitious plans for progress in Brexit negotiations in the coming weeks. Arriving at th...
     topic  : worldnews
     speaker: nan

  ── Fake (label=1) ── source: ISOT
     title  : TELL-ALL BOOK Exposes What A Nightmare Hillary Clinton Was To White House Staff…"Jekyll and Hyde"
     content: We ve heard about Hillary Clinton s bad behavior before but this tell-all book goes right into details 

---
## Cell 15 — JSON Export Preview
Shows how a row maps to the target JSON schema used by the preprocessing pipeline.

In [18]:
import json

# ── Sample row → JSON record ───────────────────────────────────────────────────
sample = df_check.sample(1, random_state=7).iloc[0]
record = {
    "id"             : sample['id'],
    "title"          : sample['title'],
    "content"        : sample['content'],
    "label"          : int(sample['label']),
    "label_text"     : sample['label_text'],
    "label_original" : sample['label_original'],
    "source_dataset" : sample['source_dataset'],
    "metadata": {
        "topic"  : sample['topic'],
        "url"    : sample['url'],
        "speaker": sample['speaker'],
    }
}
print('Sample JSON record:')
print(json.dumps(record, indent=2, ensure_ascii=False))

# ── Optional: Export full dataset as JSON Lines ────────────────────────────────
# Uncomment to save a .jsonl file (one JSON object per line):
# jsonl_path = os.path.join(DATA_DIR, 'Dataset_Clean.jsonl')
# with open(jsonl_path, 'w', encoding='utf-8') as f:
#     for _, row in df_check.iterrows():
#         rec = {
#             'id': row['id'], 'title': row['title'], 'content': row['content'],
#             'label': int(row['label']), 'label_text': row['label_text'],
#             'label_original': row['label_original'], 'source_dataset': row['source_dataset'],
#             'metadata': {'topic': row['topic'], 'url': row['url'], 'speaker': row['speaker']}
#         }
#         f.write(json.dumps(rec, ensure_ascii=False) + '\n')
# print(f'\nJSON Lines saved → {jsonl_path}')

Sample JSON record:
{
  "id": "LIAR_008089",
  "title": "Government subsidies for renewable energies such as wind and solar are 100 times greater than those given to gas and coal, and 50 times greater than what the nuclear industry enjoys.",
  "content": "Government subsidies for renewable energies such as wind and solar are 100 times greater than those given to gas and coal, and 50 times greater than what the nuclear industry enjoys.",
  "label": 3,
  "label_text": "Bias",
  "label_original": "half-true",
  "source_dataset": "LIAR",
  "metadata": {
    "topic": "energy",
    "url": NaN,
    "speaker": "tom-fanning"
  }
}
